In [ ]:
# Inspect the entire dolchid dataset saved to my google drive
from google.colab import drive
drive.mount('/content/drive')

import zipfile

zip_path = "/content/drive/MyDrive/DOLCHID/DOLCHID.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

print("Total files:", len(files))
print("\nFirst 30 files:")
for f in files[:30]:
    print(f)

Mounted at /content/drive
Total files: 1053

First 30 files:
DOLCHID/
DOLCHID/cbct_image/
DOLCHID/cbct_image/AME_10_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_11_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_12_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_13_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_14_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_15_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_16_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_17_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_18_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_19_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_1_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_20_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_21_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_22_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_23_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_24_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_25_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_26_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_27_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_28_CBCT_Image.nii.gz
DOLCHID/cbct_image/AME_29_CBCT_

In [ ]:

# Count pathology class distribution
import zipfile
from collections import Counter

zip_path = "/content/drive/MyDrive/DOLCHID/DOLCHID.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    images = [
        n for n in names
        if n.startswith("DOLCHID/cbct_image/")
        and n.endswith(".nii.gz")
    ]

    pathology_counts = Counter(
        n.split("/")[-1].split("_")[0]
        for n in images
    )

    print("Total CBCT scans:", len(images))
    print("\nScans per pathology:")
    for pathology, count in sorted(pathology_counts.items()):
        print(f"{pathology}: {count}")

    print("\nExample mask filenames:")
    for n in names:
        if "label" in n.lower() and n.endswith(".nii.gz"):
            print(n)
            if sum(1 for _ in []) > 10:
                break


Total CBCT scans: 262

Scans per pathology:
AME: 72
DC: 44
KCOT: 92
RC: 54

Example mask filenames:
DOLCHID/cbct_label/AME_10_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_11_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_12_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_13_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_14_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_15_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_16_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_17_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_18_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_19_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_1_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_20_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_21_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_22_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_23_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_24_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_25_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_26_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_27_CBCT_Label.nii.gz
DOLCHID/cbct_label/AME_28_CBCT_Label.nii.gz
DOLCHID/cbct_label/AM

In [ ]:

# Choose randomly a subset of 140 samples, with 35 images from each pathology
import os
import zipfile
import random
import shutil
from collections import defaultdict
from pathlib import PurePosixPath

zip_path = "/content/drive/MyDrive/DOLCHID/DOLCHID.zip"
output_dir = "/content/drive/MyDrive/DOLCHID_140"

random.seed(42)

pathologies = ["AME", "DC", "KCOT", "RC"]
samples_per_pathology = 35

os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    # Find all CBCT images
    images_by_pathology = defaultdict(list)

    for name in names:
        if not name.endswith(".nii.gz"):
            continue

        if "/cbct_image/" not in "/" + name:
            continue

        filename = PurePosixPath(name).name
        pathology = filename.split("_")[0]

        if pathology in pathologies:
            images_by_pathology[pathology].append(name)

    selected_images = []

    for pathology in pathologies:
        available = images_by_pathology[pathology]

        if len(available) < samples_per_pathology:
            raise ValueError(
                f"Only {len(available)} scans available for {pathology}"
            )

        chosen = random.sample(available, samples_per_pathology)
        selected_images.extend(chosen)

    print("Selected scans:", len(selected_images))

    extracted = 0
    missing_labels = []

    for image_path in selected_images:
        image_name = PurePosixPath(image_path).name

        case_id = image_name.replace("_CBCT_Image.nii.gz", "")
        pathology = case_id.split("_")[0]

        label_name = f"{case_id}_CBCT_Label.nii.gz"

        matching_labels = [
            n for n in names
            if PurePosixPath(n).name == label_name
            and "/cbct_label/" in "/" + n
        ]

        if not matching_labels:
            missing_labels.append(case_id)
            continue

        label_path = matching_labels[0]

        for archive_path, subfolder in [
            (image_path, "cbct_image"),
            (label_path, "cbct_label")
        ]:
            destination_folder = os.path.join(output_dir, subfolder)
            os.makedirs(destination_folder, exist_ok=True)

            destination = os.path.join(
                destination_folder,
                PurePosixPath(archive_path).name
            )

            if not os.path.exists(destination):
                with z.open(archive_path) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        extracted += 1
        print(f"{extracted}/140: {case_id}")

    print("\nFinished.")
    print("Complete image-label pairs:", extracted)
    print("Missing labels:", missing_labels)
    print("Output folder:", output_dir)


Selected scans: 140
1/140: AME_23
2/140: AME_13
3/140: AME_42
4/140: AME_39
5/140: AME_36
6/140: AME_26
7/140: AME_22
8/140: AME_20
9/140: AME_5
10/140: AME_12
11/140: AME_11
12/140: AME_15
13/140: AME_6
14/140: AME_9
15/140: AME_3
16/140: AME_45
17/140: AME_66
18/140: AME_7
19/140: AME_21
20/140: AME_51
21/140: AME_48
22/140: AME_50
23/140: AME_41
24/140: AME_34
25/140: AME_63
26/140: AME_71
27/140: AME_44
28/140: AME_70
29/140: AME_10
30/140: AME_1
31/140: AME_35
32/140: AME_2
33/140: AME_56
34/140: AME_19
35/140: AME_64
36/140: DC_2
37/140: DC_16
38/140: DC_15
39/140: DC_32
40/140: DC_8
41/140: DC_30
42/140: DC_4
43/140: DC_25
44/140: DC_12
45/140: DC_37
46/140: DC_17
47/140: DC_6
48/140: DC_7
49/140: DC_26
50/140: DC_19
51/140: DC_34
52/140: DC_29
53/140: DC_28
54/140: DC_20
55/140: DC_27
56/140: DC_5
57/140: DC_44
58/140: DC_42
59/140: DC_11
60/140: DC_40
61/140: DC_41
62/140: DC_9
63/140: DC_36
64/140: DC_13
65/140: DC_31
66/140: DC_14
67/140: DC_43
68/140: DC_1
69/140: DC_39
70/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [8]:
# Load the dataset and make dictionaries of the image, mask and class name
import os
import importlib
import datasetIO
importlib.reload(datasetIO)
from datasetIO import load_dolchid_dataset
output_dir = "/content/drive/MyDrive/DOLCHID_140"
data_dicts = load_dolchid_dataset(output_dir)
print(data_dicts[0])

Successfully loaded 140 DOLCHID samples.
{'image': '/content/drive/MyDrive/DOLCHID_140/cbct_image/AME_10_CBCT_Image.nii.gz', 'mask': '/content/drive/MyDrive/DOLCHID_140/cbct_mask/AME_10_CBCT_Label.nii.gz', 'class_name': 'AME', 'class_id': 0}


In [9]:
# Run the split function that splits the smaller dataset into train, val and test
# this is run only once to create the csv file containing the spliting sets
import importlib
import split
importlib.reload(split)

splitted_set = split.split_data(data_dicts, "/content/drive/MyDrive/cbct-ai/metadata.csv")
print(splitted_set['train'][0])

Total Patients: 140
Training set:   101 patients
Validation set: 18 patients
Test set: 21 patients
{'image': '/content/drive/MyDrive/DOLCHID_140/cbct_image/AME_11_CBCT_Image.nii.gz', 'mask': '/content/drive/MyDrive/DOLCHID_140/cbct_mask/AME_11_CBCT_Label.nii.gz', 'class_name': 'AME', 'class_id': 0, 'split': 'train', 'case_id': 'AME_11'}


In [12]:
# Run the actual static preprocessing (spacing, orientation) and create the actual datasets ready for segm and classification
from google.colab import drive
drive.mount("/content/drive")
!pip install -q monai nibabel pandas

!python base.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
I0000 00:00:1793363213.504123   22380 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793363215.353869   22380 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793363221.000432   22380 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=N